# Week 03 — Lines & Slopegraph: Stock Performance

A time-series dataset with five large technology companies. The analysis focuses on relative movement rather than pretending the series are directly comparable prices.

## Task 1 — Highlight one stock
Normalize each company to 100 at the first date, then highlight Microsoft against the other series.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("../data/stocks.csv", parse_dates=["date"])
companies = ["GOOG","AAPL","AMZN","FB","NFLX","MSFT"]
long = df.melt("date", var_name="company", value_name="price")
base = long.groupby("company")["price"].transform("first")
long["index_100"] = long["price"] / base * 100

fig = go.Figure()
for c in companies:
    d = long[long.company == c]
    fig.add_trace(go.Scatter(
        x=d.date, y=d.index_100, mode="lines",
        name=c, line=dict(width=4 if c=="MSFT" else 1.5,
                          color="#1f77b4" if c=="MSFT" else "#cccccc")
    ))
fig.update_layout(title="Microsoft finishes well above its starting level",
                  yaxis_title="Indexed value (start = 100)", xaxis_title="Date",
                  plot_bgcolor="white", paper_bgcolor="white")
fig.show()

## Task 2 — Slopegraph
Compare the first and last observed values after indexing. Label the start and end points.

In [ ]:
start = long.sort_values("date").groupby("company").first()["index_100"]
end = long.sort_values("date").groupby("company").last()["index_100"]
slope = pd.DataFrame({"start":start, "end":end}).sort_values("end")

fig = go.Figure()
for company, r in slope.iterrows():
    fig.add_trace(go.Scatter(x=["Start","End"], y=[r.start,r.end],
                             mode="lines+markers", name=company))
    fig.add_annotation(x="End", y=r.end, text=f"{company} {r.end:.0f}",
                       showarrow=False, xshift=45)
fig.update_layout(title="Every tracked stock gained, but the scale of gains differs",
                  showlegend=False, yaxis_title="Indexed value", xaxis_title="",
                  plot_bgcolor="white", paper_bgcolor="white")
fig.show()